In [ ]:
import pandas as pd
df = pd.read_json("hf://datasets/adamwhite625/vietnam-legal-qa/ready_to_import_dataset.json")

In [ ]:
print(df.head())
print(type(df))

In [ ]:
print(df.info())

In [ ]:
df.describe().T.drop(columns=["top", "freq"]).to_latex()

In [ ]:
df['id'].nunique(), df['law_id'].nunique()


In [ ]:
df.groupby('id')['law_id'].nunique().sort_values(ascending=False)

### Xây dựng kho tri thức

In [ ]:
import os
import json
#print(os.listdir())
with open('ready_to_import_dataset.json','r',encoding='utf-8') as f:
  raw_data = json.load(f)

In [ ]:
print(type(raw_data))
for i in range(3):
  print(raw_data[i].keys())


Chuẩn hóa thành slug dạng snake_case

In [ ]:
import re
import unicodedata

In [ ]:
text = 'Hôm nay trời đẹp'
print(repr(text))
a = unicodedata.normalize('NFKD', text)
for c in a:
  print(c, hex(ord(c)), unicodedata.name(c))

In [ ]:
from slugify import slugify

text = "Học lập trình Python cùng đồng nghiệp!"
slug = slugify(text)
print(slug)

In [ ]:
def build_corpus(input_file: str, output_file: str) -> list:
  with open(input_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
  corpus = []
  seen_laws = set()
  for item in raw_data:
    law_name = item.get('law_name',"").strip()
    law_id = item.get('law_id', "").strip()
    law_content = item.get('law_content',"").strip()

    unique_key = (law_name,law_id)
    slug = lambda text: slugify(text)
    if unique_key not in seen_laws:
      seen_laws.add(unique_key)
      doc_id = f'{slug(law_name)}_{slug(law_id)}'
      corpus.append({
          'doc_id':doc_id,
          'law_name': law_name,
          'law_id': law_id,
          'law_content': law_content
      })
  with open(output_file, 'w', encoding = 'utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
  return corpus

### Câu hỏi
Slug là gì ? tại sao cần phải slug


In [ ]:
my_dict = {'law_name': 'Điều 172'}
my_list = []
my_list.append(my_dict)
for item in my_list:
  law_name = item.get('law_name',"").strip()
  print(law_name)

In [ ]:
corpus = build_corpus('ready_to_import_dataset.json', 'corpus.json')

In [ ]:
import pandas as pd
df_corpus = pd.read_json('corpus.json')

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
df_corpus["char_length"] = df_corpus["law_content"].apply(len)
# - Số từ (cắt theo khoảng trắng)
df_corpus["word_length"] = df_corpus["law_content"].apply(
    lambda x: len(x.split())
)
# - Số dòng / đoạn phân tách
df_corpus["num_clauses"] = df_corpus["law_content"].apply(
    lambda x: len([line for line in x.split("\n") if line.strip()])
)

# 3. Bảng thống kê chi tiết từng điều luật (sắp xếp theo điều dài nhất giảm dần)
summary_table = df_corpus[
    ["doc_id", "law_id", "law_name", "word_length", "char_length"]
].sort_values(by="word_length", ascending=False)

print("=== TOP 10 ĐIỀU LUẬT DÀI NHẤT ===")
print(summary_table.head(10).to_string(index=False))

print("\n=== TOP 5 ĐIỀU LUẬT NGẮN NHẤT ===")
print(summary_table.tail(5).to_string(index=False))

# 4. Thống kê mô tả tổng quan (Describe)
print("\n=== THỐNG KÊ MÔ TẢ ĐỘ DÀI TOÀN BỘ CORPUS (SỐ TỪ) ===")
stats_df = df_corpus[["word_length"]].describe().to_latex()
print(stats_df)

# 5. Vẽ biểu đồ phân bố độ dài từ
plt.figure(figsize=(10, 4))
sns.histplot(df_corpus["word_length"], kde=True, bins=20, color="teal")
plt.title("Phân bố độ dài (Số từ) của các Điều luật trong Corpus")
plt.xlabel("Số từ")
plt.ylabel("Số lượng Điều luật")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()